Electricity data **DELETED**

In [ ]:
import pandas as pd
from pathlib import Path

carriers_path = 'plants_data/carriers/'

belgium = pd.read_csv(Path(carriers_path) / 'Belgium_2024.csv', sep=',')
germany = pd.read_csv(Path(carriers_path) / 'Germany_2024.csv', sep=',')
netherlands = pd.read_csv(Path(carriers_path) / 'Netherlands_2024.csv', sep=',')
# keep colums MTU (CET/CEST), Day-ahead Price (EUR/MWh)
belgium = belgium[['MTU (CET/CEST)', 'Day-ahead Price (EUR/MWh)']]
germany = germany[['MTU (CET/CEST)', 'Day-ahead Price (EUR/MWh)']]
netherlands = netherlands[['MTU (CET/CEST)', 'Day-ahead Price (EUR/MWh)']]

# remove anything after the second space included
belgium['MTU (CET/CEST)'] = belgium['MTU (CET/CEST)'].str.split(' ').str[0:2].str.join(' ')
germany['MTU (CET/CEST)'] = germany['MTU (CET/CEST)'].str.split(' ').str[0:2].str.join(' ')
netherlands['MTU (CET/CEST)'] = netherlands['MTU (CET/CEST)'].str.split(' ').str[0:2].str.join(' ')
# convert MTU (CET/CEST) column to datetime format
belgium['MTU (CET/CEST)'] = pd.to_datetime(belgium['MTU (CET/CEST)'], format='%d/%m/%Y %H:%M:%S')
germany['MTU (CET/CEST)'] = pd.to_datetime(germany['MTU (CET/CEST)'], format='%d/%m/%Y %H:%M:%S')
netherlands['MTU (CET/CEST)'] = pd.to_datetime(netherlands['MTU (CET/CEST)'], format='%d/%m/%Y %H:%M:%S')

# if datetime are repeated, keep the first occurrence
belgium = belgium.drop_duplicates(subset=['MTU (CET/CEST)'], keep='first')
germany = germany.drop_duplicates(subset=['MTU (CET/CEST)'], keep='first')
netherlands = netherlands.drop_duplicates(subset=['MTU (CET/CEST)'], keep='first')

# if not hourly data, resample to hourly data by taking the mean of each hour
belgium = belgium.set_index('MTU (CET/CEST)').resample('h').mean().reset_index()
germany = germany.set_index('MTU (CET/CEST)').resample('h').mean().reset_index()
netherlands = netherlands.set_index('MTU (CET/CEST)').resample('h').mean().reset_index()

full_index = pd.date_range(start='2024-01-01 00:00:00', end='2024-12-31 23:00:00', freq='h')
# reindex to have a full year from 2024-01-01 to 2024-12-31 23:00:00
belgium = belgium.set_index('MTU (CET/CEST)').reindex(full_index).rename_axis('MTU (CET/CEST)').reset_index()
germany = germany.set_index('MTU (CET/CEST)').reindex(full_index).rename_axis('MTU (CET/CEST)').reset_index()
netherlands = netherlands.set_index('MTU (CET/CEST)').reindex(full_index).rename_axis('MTU (CET/CEST)').reset_index()

# output cleaned files
output_file = 'electricity_prices_2024.csv'
with open(Path(carriers_path) / output_file, 'w') as f:
    f.write('datetime,Belgium (EUR/MWh),Germany (EUR/MWh),Netherlands (EUR/MWh)\n')
    for i in range(len(full_index)):
        f.write(f"{full_index[i].strftime('%Y-%m-%d %H:%M:%S')},{belgium['Day-ahead Price (EUR/MWh)'].iloc[i]},{germany['Day-ahead Price (EUR/MWh)'].iloc[i]},{netherlands['Day-ahead Price (EUR/MWh)'].iloc[i]}\n")


Electricity data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'
scenario = 'flexible'
year = 2030

country_dict = {
    'Belgium': 'BE',
    'Germany': 'DE',
    'Netherlands': 'NL'}

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)
prices_data = pd.read_csv(f'plants_data/electricity_data/prices_AC_allin_{scenario}_{year}.csv', sep=',', index_col=0)

# --- Indice orario per il 2025 ---
full_index = pd.date_range(f'2025-01-01 00:00:00', f'2025-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'electricity.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:
        country = europe_plants.loc[plant, 'country']
        country_code = country_dict[country]

        # --- Prezzi ---
        prices = prices_data[country_code].values
        prices = np.clip(prices, 0.01, None)  # evitiamo negativi

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': prices,
            'demand': 0.0,                # MW costante
            'export_price': 0.01,         # EUR/MWh
            'import_limit': 1e6,          # MW
            'export_limit': 1e6           # MW
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')


Hydrogen data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2025 ---
full_index = pd.date_range('2025-01-01 00:00:00', '2025-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'hydrogen.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': np.nan,
            'demand': 0.0,
            'export_price': 0,
            'import_limit': 0,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')


Ammonia

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2025 ---
full_index = pd.date_range('2025-01-01 00:00:00', '2025-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'ammonia.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        demand = europe_plants.loc[plant, 'ammonia'] if 'ammonia' in europe_plants.columns else 0.0

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': np.nan,
            'demand': demand/8760*1000, # convertiamo in hourly demand and t (from kt/y)
            'export_price': 0,
            'import_limit': 0,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')


Ethylene

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2025 ---
full_index = pd.date_range('2025-01-01 00:00:00', '2025-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'ethylene.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        demand = europe_plants.loc[plant, 'ethylene'] if 'ethylene' in europe_plants.columns else 0.0

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': np.nan,
            'demand': demand/8760*1000, # convertiamo in hourly demand and t (from kt/y)
            'export_price': 0,
            'import_limit': 0,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Methanol

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2025 ---
full_index = pd.date_range('2025-01-01 00:00:00', '2025-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'methanol.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        demand = europe_plants.loc[plant, 'methanol'] if 'methanol' in europe_plants.columns else 0.0

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': np.nan,
            'demand': demand/8760*1000, # convertiamo in hourly demand and t (from kt/y)
            'export_price': 0,
            'import_limit': 0,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Propylene

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2025 ---
full_index = pd.date_range('2025-01-01 00:00:00', '2025-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'propylene.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        demand = europe_plants.loc[plant, 'propylene'] if 'propylene' in europe_plants.columns else 0.0

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': np.nan,
            'demand': demand/8760*1000, # convertiamo in hourly demand and t (from kt/y)
            'export_price': 0,
            'import_limit': 0,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Carbon dioxide

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2025 ---
full_index = pd.date_range('2025-01-01 00:00:00', '2025-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'CO2.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': 618, # considering DAC cost
            'demand': np.nan,
            'export_price': 0,
            'import_limit': 1e6,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')


Methane

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2025 ---
full_index = pd.date_range('2025-01-01 00:00:00', '2025-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'methane.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': 56,
            'demand': 0,
            'export_price': 0,
            'import_limit': 1e6,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Heat

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2025 ---
full_index = pd.date_range('2025-01-01 00:00:00', '2025-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'heat.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': np.nan,
            'demand': 0.0,
            'export_price': 0.01,
            'import_limit': 0,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Steam

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2025 ---
full_index = pd.date_range('2025-01-01 00:00:00', '2025-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'steam.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': np.nan,
            'demand': 0.0,
            'export_price': 0,
            'import_limit': 0,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

HBfeed

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2025 ---
full_index = pd.date_range('2025-01-01 00:00:00', '2025-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'HBfeed.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': np.nan,
            'demand': 0.0,
            'export_price': 0.0,
            'import_limit': 0,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

CO2captured

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2025 ---
full_index = pd.date_range('2025-01-01 00:00:00', '2025-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'CO2captured.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:
        if plant == 'NL1':
            # --- Creazione DataFrame per l’impianto ---
            df = pd.DataFrame({
                'import_price': 100,
                'demand': 0.0,
                'export_price': -0.01,
                'import_limit': 1e6,
                'export_limit': 1e6
            }, index=full_index)
        else:
        # --- Creazione DataFrame per l’impianto ---
            df = pd.DataFrame({
                'import_price': np.nan,
                'demand': 0.0,
                'export_price': 0.0,
                'import_limit': 0,
                'export_limit': 1e6
            }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Nitrogen

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2025 ---
full_index = pd.date_range('2025-01-01 00:00:00', '2025-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'nitrogen.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': np.nan,
            'demand': 0.0,
            'export_price': 0.01,
            'import_limit': 0,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Naphtha

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2025 ---
full_index = pd.date_range('2025-01-01 00:00:00', '2025-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'naphtha.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': 732,
            'demand': 0,
            'export_price': 0,
            'import_limit': 1e6,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Olefins

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2025 ---
full_index = pd.date_range('2025-01-01 00:00:00', '2025-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'olefins.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': np.nan,
            'demand': 0.0,
            'export_price': 0.01,
            'import_limit': 0,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Crackergas

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2025 ---
full_index = pd.date_range('2025-01-01 00:00:00', '2025-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'crackergas.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': np.nan,
            'demand': 0.0,
            'export_price': 0.01,
            'import_limit': 0,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Syngas

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2025 ---
full_index = pd.date_range('2025-01-01 00:00:00', '2025-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'syngas.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': np.nan,
            'demand': 0.0,
            'export_price': 0.01,
            'import_limit': 0,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

MPW (Mixed Plastic Waste)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2025 ---
full_index = pd.date_range('2025-01-01 00:00:00', '2025-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'MPW.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': 780,
            'demand': 0.0,
            'export_price': 0.01,
            'import_limit': 1e6,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Ethanol

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2025 ---
full_index = pd.date_range('2025-01-01 00:00:00', '2025-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'ethanol.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': 1734,
            'demand': 0.0,
            'export_price': 0.01,
            'import_limit': 1e6,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Propane **DELETED**

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2025 ---
full_index = pd.date_range('2025-01-01 00:00:00', '2025-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'propane.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': 1473,
            'demand': 0.0,
            'export_price': 0.01,
            'import_limit': 1e6,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Feedgas

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2025 ---
full_index = pd.date_range('2025-01-01 00:00:00', '2025-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'feedgas.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': np.nan,
            'demand': 0.0,
            'export_price': 0.01,
            'import_limit': 0,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

Bio-methane

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

carriers_path = 'plants_data/carriers/'

# --- Lettura dati ---
europe_plants = pd.read_csv('plants_data/plants_clusters_summary.csv', sep=',', index_col=0)

# --- Indice orario per il 2025 ---
full_index = pd.date_range('2025-01-01 00:00:00', '2025-12-31 23:00:00', freq='h')

# --- File Excel di output ---
with pd.ExcelWriter(Path(carriers_path) / 'methane-bio.xlsx', engine='openpyxl', mode='w') as writer:
    for plant in europe_plants.index:

        # --- Creazione DataFrame per l’impianto ---
        df = pd.DataFrame({
            'import_price': 1833, # EUR/t (from 141 €/MWh from Julia)
            'demand': 0.0,
            'export_price': 0.01,
            'import_limit': 1e6,
            'export_limit': 1e6
        }, index=full_index)

        # --- Salvataggio in foglio dedicato ---
        df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

# Electricity inputs
electricity_dict = {
    'scenario': 'flexible', # flexible, flexible-moderate, rigid, rigid-tes
    'year': '2030', # 2030, 2040, 2050
    'import_limit': 1e6, # MW
    'export_price': 0.01, # EUR/MWh
    'export_limit': 1e6, # MW
    'demand': 0.0 # MW
}

# Hydrogen inputs
hydrogen_dict = {
    'carrier': 'hydrogen',
    'import_price': np.nan, # EUR/t
    'demand': 0.0, # t/h
    'export_price': 0.01, # EUR/t
    'import_limit': 0, # t/h
    'export_limit': 1e6, # t/h
}

# Ammonia inputs
ammonia_dict = {
    'carrier': 'ammonia',
    'import_price': np.nan, # EUR/t
    'export_price': 0.01, # EUR/t
    'import_limit': 0, # t/h
    'export_limit': 1e6, # t/h
}

# Ethylene inputs
ethylene_dict = {
    'carrier': 'ethylene',
    'import_price': np.nan, # EUR/t
    'demand': 0.0, # t/h
    'export_price': 0.01, # EUR/t
    'import_limit': 0, # t/h
    'export_limit': 1e6, # t/h
}

# Propylene inputs
propylene_dict = {
    'carrier': 'propylene',
    'import_price': np.nan, # EUR/t
    'demand': 0.0, # t/h
    'export_price': 0.01, # EUR/t
    'import_limit': 0, # t/h
    'export_limit': 1e6, # t/h
}

# Methanol inputs
methanol_dict = {
    'carrier': 'methanol',
    'import_price': np.nan, # EUR/t
    'demand': 0.0, # t/h
    'export_price': 0.01, # EUR/t
    'import_limit': 0, # t/h
    'export_limit': 1e6, # t/h
}

# CO2 inputs
CO2_dict = {
    'carrier': 'CO2',
    'import_price': 618, # EUR/t (considering DAC cost)
    'demand': np.nan, # t/h
    'export_price': 0, # EUR/t
    'import_limit': 1e6, # t/h
    'export_limit': 1e6, # t/h
}

# Methane inputs
methane_dict = {
    'carrier': 'methane',
    'import_price': 56, # EUR/t
    'demand': 0, # t/h
    'export_price': 0, # EUR/t
    'import_limit': 1e6, # t/h
    'export_limit': 1e6, # t/h
}

# Heat inputs
heat_dict = {
    'carrier': 'heat',
    'import_price': np.nan, # EUR/MWh
    'demand': 0.0, # MW
    'export_price': 0.01, # EUR/MWh
    'import_limit': 0, # MW
    'export_limit': 1e6, # MW
}

# Steam inputs
steam_dict = {
    'carrier': 'steam',
    'import_price': np.nan, # EUR/MWh
    'demand': 0.0, # MW
    'export_price': 0, # EUR/MWh
    'import_limit': 0, # MW
    'export_limit': 1e6, # MW
}

# biomass feedstock inputs
biomass_dict = {
    'carrier': 'biomass',
    'import_price': np.nan, # EUR/t
    'demand': 0.0, # t/h
    'export_price': 0.01, # EUR/t
    'import_limit': 10000, # t/h
    'export_limit': 1e6, # t/h
}

methane_bio_dict = {
    'carrier': 'methane_bio',
    'import_price': 1833, # EUR/t (from 141 €/MWh from Julia)
    'demand': 0.0, # t/h
    'export_price': 0.01, # EUR/t
    'import_limit': 1e6, # t/h
    'export_limit': 1e6, # t/h
}

feedgas_dict = {
    'carrier': 'feedgas',
    'import_price': np.nan, # EUR/t
    'demand': 0.0, # t/h
    'export_price': 0.01, # EUR/t
    'import_limit': 0, # t/h
    'export_limit': 1e6, # t/h
}

syngas_dict = {
    'carrier': 'syngas',
    'import_price': np.nan, # EUR/t
    'demand': 0.0, # t/h
    'export_price': 0.01, # EUR/t
    'import_limit': 0, # t/h
    'export_limit': 1e6, # t/h
}

crackergas_dict = {
    'carrier': 'crackergas',
    'import_price': np.nan, # EUR/t
    'demand': 0.0, # t/h
    'export_price': 0.01, # EUR/t
    'import_limit': 0, # t/h
    'export_limit': 1e6, # t/h
}

olefins_dict = {
    'carrier': 'olefins',
    'import_price': np.nan, # EUR/t
    'demand': 0.0, # t/h
    'export_price': 0.01, # EUR/t
    'import_limit': 0, # t/h
    'export_limit': 1e6, # t/h
}

naphtha_dict = {
    'carrier': 'naphtha',
    'import_price': 732, # EUR/t
    'demand': 0.0, # t/h
    'export_price': 0.01, # EUR/t
    'import_limit': 1e6, # t/h
    'export_limit': 1e6, # t/h
}

HB_feed_dict = {
    'carrier': 'HBfeed',
    'import_price': np.nan, # EUR/t
    'demand': 0.0, # t/h
    'export_price': 0.01, # EUR/t
    'import_limit': 0, # t/h
    'export_limit': 1e6, # t/h
}

nitrogen_dict = {
    'carrier': 'nitrogen',
    'import_price': np.nan, # EUR/t
    'demand': 0.0, # t/h
    'export_price': 0.01, # EUR/t
    'import_limit': 0, # t/h
    'export_limit': 1e6, # t/h
}


def electricity_carrier(nodes, full_index, carriers_path, electricity_dict):
    country_dict = {
    'Belgium': 'BE',
    'Germany': 'DE',
    'Netherlands': 'NL'}

    # Read data
    prices_data = pd.read_csv(f'plants_data/electricity_data/prices_AC_allin_{electricity_dict["scenario"]}_{electricity_dict["year"]}.csv', sep=',', index_col=0)

    # Write excel file
    with pd.ExcelWriter(Path(carriers_path) / 'electricity.xlsx', engine='openpyxl', mode='w') as writer:
        for plant in nodes.index:
            country = nodes.loc[plant, 'country']
            country_code = country_dict[country]

            # Prices
            prices = prices_data[country_code].values
            prices = np.clip(prices, 0.01, None)  # avoid negative prices

            # Create DataFrame for the plant
            df = pd.DataFrame({
                'import_price': prices,
                'demand': electricity_dict['demand'],                # constant MW
                'export_price': electricity_dict['export_price'],         # EUR/MWh
                'import_limit': electricity_dict['import_limit'],          # MW
                'export_limit': electricity_dict['export_limit']           # MW
            }, index=full_index)

            # Save to dedicated sheet
            df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

    return

def biomass_carrier(nodes, full_index, carriers_path, biomass_dict):

    country_dict = {
    'Belgium': 'BE',
    'Germany': 'DE',
    'Netherlands': 'NL'}

    prices_data = pd.read_csv(f'plants_data\dry_biomass_data.csv', index_col=0, sep=',')

    with pd.ExcelWriter(Path(carriers_path) / 'biomass.xlsx', engine='openpyxl', mode='w') as writer:
        for plant in nodes.index:

            country = nodes.loc[plant, 'country']
            country_code = country_dict[country]

            # Prices
            prices = prices_data.loc[country_code].values
            prices = np.clip(prices, 0.01, None)  # avoid negative prices
            prices = prices *18.5 / 3.6 # convert from EUR/MWh to EUR/t (considering 18.5 GJ/t)

            # Create DataFrame for the plant
            df = pd.DataFrame({
                'import_price': prices, # EUR/t
                'demand': biomass_dict['demand'], # t/h
                'export_price': biomass_dict['export_price'], # EUR/t
                'import_limit': biomass_dict['import_limit'], # t/h
                'export_limit': biomass_dict['export_limit'] # t/h
            }, index=full_index)

            # Save to dedicated sheet
            df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

    return

def carriers(nodes, full_index, carriers_path, dict):

    carrier = dict['carrier']

    with pd.ExcelWriter(Path(carriers_path) / f'{carrier}.xlsx', engine='openpyxl', mode='w') as writer:
        for plant in nodes.index:

            demand = nodes.loc[plant, carrier]/8760*1000 if carrier in nodes.columns else dict['demand']

            # Create DataFrame for the plant
            df = pd.DataFrame({
                'import_price': dict['import_price'], # EUR/t
                'demand': demand, # t/h
                'export_price': dict['export_price'], # EUR/t
                'import_limit': dict['import_limit'], # t/h
                'export_limit': dict['export_limit'] # t/h
            }, index=full_index)

            # Save to dedicated sheet
            df.to_excel(writer, sheet_name=plant[:31], index_label='datetime')

    return

try:
    # Questa variabile esiste solo dentro i notebook di VS Code
    notebook_path = __vsc_ipynb_file__
    current_folder = os.path.dirname(notebook_path)
except NameError:
    # Fallback nel caso la variabile non esistesse (es. fuori da VS Code)
    current_folder = os.getcwd()
    print('Warning: __vsc_ipynb_file__ not found. Using current working directory as base path.')

nodes = pd.read_csv(current_folder + '/plants_clusters_summary.csv', sep=',', index_col=0)    
full_index = pd.date_range(start='2025-01-01 00:00:00', end='2025-12-31 23:00:00', freq='h')
# carriers_path = 'case_studies/' + case_study + '/carriers/'

# electricity_carrier(nodes, full_index, carriers_path, electricity_dict)
# biomass_carrier(nodes, full_index, carriers_path, biomass_dict)
# carriers(nodes, full_index, carriers_path, hydrogen_dict)
# carriers(nodes, full_index, carriers_path, ammonia_dict)
# carriers(nodes, full_index, carriers_path, ethylene_dict)
# carriers(nodes, full_index, carriers_path, propylene_dict)
# carriers(nodes, full_index, carriers_path, methanol_dict)
# carriers(nodes, full_index, carriers_path, CO2_dict)
# carriers(nodes, full_index, carriers_path, methane_dict)
# carriers(nodes, full_index, carriers_path, heat_dict)
# carriers(nodes, full_index, carriers_path, steam_dict)
# carriers(nodes, full_index, carriers_path, HB_feed_dict)
# carriers(nodes, full_index, carriers_path, nitrogen_dict)
# carriers(nodes, full_index, carriers_path, naphtha_dict)
# carriers(nodes, full_index, carriers_path, olefins_dict)
# carriers(nodes, full_index, carriers_path, crackergas_dict)
# carriers(nodes, full_index, carriers_path, syngas_dict)
# carriers(nodes, full_index, carriers_path, feedgas_dict)
# carriers(nodes, full_index, carriers_path, methane_bio_dict)
